In [29]:
import chromadb
import fitz # PyMuPDF
from langchain_core.documents import Document
import os
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
import google.generativeai as genai

In [30]:
load_dotenv()

True

In [31]:
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

In [32]:
# STEP 1: Persistent Client + Collection

# Create persistent client (this folder will store DB)
client = chromadb.PersistentClient(path="./chromadb")

# Create or get collection
collection = client.get_or_create_collection(
    name="gov_schemes_rag"
)

# Check collections
print(client.list_collections())

[Collection(name=gov_schemes_rag)]


In [33]:
# STEP 2: TXT Loader (Final)

def load_txt_files(folder_path):
    documents = []
    
    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            file_path = os.path.join(folder_path, file)
            
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()
                
                text = text.strip()
                
                if len(text) < 50:
                    continue
                
                doc = Document(
                    page_content=text,
                    metadata={
                        "source": file,
                        "type": "government_scheme",
                        "category": "education/science"
                    }
                )
                
                documents.append(doc)
            
            except Exception as e:
                print(f"Error loading {file}: {e}")
    
    return documents

In [34]:
docs = load_txt_files("../data/raw/rag_scheme_txts")

print("Docs count:", len(docs))

if len(docs) > 0:
    print(docs[0].page_content[:500])

Docs count: 6
Scheme Name: Vigyan Dhara

Source: Department of Science and Technology (DST), Government of India

Description:
Vigyan Dhara is a unified central sector scheme that merges multiple science and technology programs to strengthen India's research and innovation ecosystem.

Objective:
- Promote science and technology capacity building
- Enhance research and innovation capabilities
- Strengthen national R&D infrastructure
- Support collaboration across academia, government, and industry

Components:


In [35]:
# STEP 3: Hybrid Splitter

def split_documents(documents):
    split_docs = []
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=50
    )
    
    for doc in documents:
        text = doc.page_content
        
        # Try structured splitting using section headers
        sections = re.split(
            r"(Scheme Name:|Description:|Objective:|Eligibility:|Benefits:|Application:|Components:)",
            text
        )
        
        # If structured sections found
        if len(sections) > 3:
            for i in range(1, len(sections), 2):
                try:
                    section_title = sections[i].strip()
                    section_content = sections[i+1].strip()
                    
                    chunk_text = f"{section_title}\n{section_content}"
                    
                    split_docs.append(
                        Document(
                            page_content=chunk_text,
                            metadata={
                                **doc.metadata,
                                "section": section_title
                            }
                        )
                    )
                except:
                    continue
        
        else:
            # Fallback to semantic chunking
            chunks = splitter.split_text(text)
            
            for chunk in chunks:
                split_docs.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata
                    )
                )
    
    return split_docs

In [36]:
split_docs = split_documents(docs)

print("Chunks:", len(split_docs))
print(split_docs[0].page_content[:300])
print(split_docs[0].metadata)

Chunks: 41
Scheme Name:
Vigyan Dhara

Source: Department of Science and Technology (DST), Government of India
{'source': 'vigyan_dhara.txt', 'type': 'government_scheme', 'category': 'education/science', 'section': 'Scheme Name:'}


In [37]:
# STEP 4: Embeddings + Store in Chroma


print("Loading embedding model...")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"
)

print("Embedding model loaded")

# Prepare data for Chroma
documents_text = [doc.page_content for doc in split_docs]
metadatas = [doc.metadata for doc in split_docs]
ids = [str(i) for i in range(len(split_docs))]

# Add to collection
collection.add(
    documents=documents_text,
    metadatas=metadatas,
    ids=ids
)

print("Stored in Chroma:", len(documents_text))

Loading embedding model...
Embedding model loaded
Stored in Chroma: 41


In [ ]:
def retrieve_docs(query, k=5):
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    
    # prioritize same source
    main_source = metadatas[0]["source"]
    
    filtered = [
        (doc, meta) for doc, meta in zip(documents, metadatas)
        if meta["source"] == main_source
    ]
    
    if len(filtered) >= 2:
        docs, metas = zip(*filtered)
    else:
        docs, metas = documents, metadatas
    
    return docs, metas

In [39]:
query = "What are the objectives of Vigyan Dhara?"

docs, metas = retrieve_docs(query)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc[:300])
    print("Metadata:", metas[i])


--- Result 1 ---
Description:
Vigyan Dhara is a unified central sector scheme that merges multiple science and technology programs to strengthen India's research and innovation ecosystem.
Metadata: {'category': 'education/science', 'source': 'vigyan_dhara.txt', 'type': 'government_scheme', 'section': 'Description:'}

--- Result 2 ---
Scheme Name:
Vigyan Dhara

Source: Department of Science and Technology (DST), Government of India
Metadata: {'section': 'Scheme Name:', 'category': 'education/science', 'source': 'vigyan_dhara.txt', 'type': 'government_scheme'}


In [46]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025

In [47]:
model = genai.GenerativeModel("models/gemini-2.5-flash")

In [ ]:
def generate_answer(query):
    docs, metas = retrieve_docs(query)
    
    context = "\n\n".join(docs)
    
    prompt = f"""
You are an expert assistant for Indian government schemes.

Instructions:
- Answer ONLY using the provided context
- If answer is not present, say "I don't know"
- Be clear, structured, and concise
- Do not hallucinate

Context:
{context}

Question:
{query}

Answer:
"""
    
    response = model.generate_content(prompt)
    
    return response.text

In [49]:
query = "What is Vigyan Dhara?"

print(generate_answer(query))

Vigyan Dhara is a unified central sector scheme that merges multiple science and technology programs to strengthen India's research and innovation ecosystem.
